# Download multiple meteoscreening variables

**notebook version**: `3` (4 Jul 2024)  
**new in this version**: added check if downloaded data is indeed in 30MIN time resolution

- This notebook can be used to download data from the database `InfluxDB`
- Data are stored to a `.csv` file in this folder

</br>

# Info about data sources of variables

- `TA`: NABEL (2004-2018), mst (2004-2021), diive (2022-2025)
- `SW_IN`: NABEL (2004-2018), mst (2005-2021), diive (2022-2025)
- `RH`: NABEL (2004-2018), mst (2004-2021), diive (2022-2025)

Legend:
- NABEL ... Data from [NABEL](https://www.bafu.admin.ch/bafu/en/home/topics/air/luftbelastung/national-air-pollution-monitoring-network--nabel-.html), meteoscreening with [diive](https://github.com/holukas/diive)
- mst ... Data from ETH, meteoscreening with the now deprecated MeteoscreeningTool
- diive ... Data from ETH, meteoscreening with diive

</br>

# Settings

## Data settings

In [1]:
DIRCONF = r'F:\Sync\luhk_work\20 - CODING\22 - POET\configs'
# DIRCONF = r'P:\Flux\RDS_calculations\_scripts\_configs\configs'  # Folder with configuration files: needed e.g. for connection to database
TIMEZONE_OFFSET_TO_UTC_HOURS = 1  # Timezone, e.g. "1" is translated to timezone "UTC+01:00" (CET, winter time)
REQUIRED_TIME_RESOLUTION = '30min'  # 30MIN time resolution
SITE_LAT = 47.478333   # CH-LAE
SITE_LON = 8.364389  # CH-LAE

## Imports

In [2]:
from datetime import datetime
from pathlib import Path
import importlib.metadata
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme('notebook')
from diive.core.plotting.timeseries import TimeSeries
from dbc_influxdb import dbcInflux
import diive as dv
from diive.core.plotting.heatmap_datetime import HeatmapDateTime
from diive.core.times.times import DetectFrequency
from diive.core.times.times import TimestampSanitizer
from diive.core.io.files import save_parquet
from diive.pkgs.createvar.potentialradiation import potrad
from diive.pkgs.gapfilling.xgboost_ts import XGBoostTS
from diive.pkgs.corrections.offsetcorrection import remove_relativehumidity_offset, remove_radiation_zero_offset
import warnings
from influxdb_client.client.warnings import MissingPivotFunction
warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)
dt_string = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
version_diive = importlib.metadata.version("diive")
print(f"diive version: v{version_diive}")
version_dbc = importlib.metadata.version("dbc_influxdb")
print(f"dbc-influxdb version: v{version_dbc}")
dbc = dbcInflux(dirconf=DIRCONF)  # Connect to database

diive version: v0.90.0
dbc-influxdb version: v0.13.1
Reading configuration files was successful.
Connection to database works.


</br>

</br>

# 🔵TA, SW_IN and RH

## `NABEL` data from `diive` meteoscreening (2004-2018)

### Download

In [3]:
%%time

BUCKET = f'ch-lae_processed'
FIELDS = ['TA_NABEL_T1_49_1', 'RH_NABEL_T1_49_1', 'SW_IN_NABEL_T1_49_1']
MEASUREMENTS = ['TA', 'RH', 'SW']
START = '2004-01-01 00:00:01'
STOP = '2019-01-01 00:00:01'
DATA_VERSION = 'meteoscreening_diive'

nabel_diive_ta_rh_swin_2004_2018, _, _ = dbc.download(
    bucket=BUCKET,
    measurements=MEASUREMENTS,
    fields=FIELDS,
    start=START,  # Download data starting with this date (the start date itself IS included),
    stop=STOP,  # Download data before this date (the stop date itself IS NOT included),
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version=DATA_VERSION
)


DOWNLOADING
    from bucket ch-lae_processed
    variables ['TA_NABEL_T1_49_1', 'RH_NABEL_T1_49_1', 'SW_IN_NABEL_T1_49_1']
    from measurements ['TA', 'RH', 'SW']
    from data version ['meteoscreening_diive']
    between 2004-01-01 00:00:01 and 2019-01-01 00:00:01
    with timezone offset to UTC of 1
Using querystring:
from(bucket: "ch-lae_processed") |> range(start: 2004-01-01T00:00:01+01:00, stop: 2019-01-01T00:00:01+01:00) |> filter(fn: (r) => r["_measurement"] == "TA" or r["_measurement"] == "RH" or r["_measurement"] == "SW") |> filter(fn: (r) => r["data_version"] == "meteoscreening_diive") |> filter(fn: (r) => r["_field"] == "TA_NABEL_T1_49_1" or r["_field"] == "RH_NABEL_T1_49_1" or r["_field"] == "SW_IN_NABEL_T1_49_1") |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
Used querystring: from(bucket: "ch-lae_processed") |> range(start: 2004-01-01T00:00:01+01:00, stop: 2019-01-01T00:00:01+01:00) |> filter(fn: (r) => r["_measurement"] == "TA" or r["_measurem

In [4]:
nabel_diive_ta_rh_swin_2004_2018

,RH_NABEL_T1_49_1,SW_IN_NABEL_T1_49_1,TA_NABEL_T1_49_1
TIMESTAMP_END,,,
2004-01-01 00:30:00,96.366667,0.0,-2.666667
2004-01-01 01:00:00,95.566667,0.0,-2.566667
2004-01-01 01:30:00,92.200000,0.0,-2.533333
2004-01-01 02:00:00,91.300000,0.0,-2.633333
2004-01-01 02:30:00,92.633333,0.0,-2.800000
...,...,...,...
2018-12-31 22:00:00,99.998000,0.0,3.193900
2018-12-31 22:30:00,99.998000,0.0,3.021733
2018-12-31 23:00:00,99.998000,0.0,2.934467


### Sanitize timestamp

In [5]:
nabel_diive_ta_rh_swin_2004_2018 = TimestampSanitizer(data=nabel_diive_ta_rh_swin_2004_2018, output_middle_timestamp=False).get()
nabel_diive_ta_rh_swin_2004_2018

,RH_NABEL_T1_49_1,SW_IN_NABEL_T1_49_1,TA_NABEL_T1_49_1
TIMESTAMP_END,,,
2004-01-01 00:30:00,96.366667,0.0,-2.666667
2004-01-01 01:00:00,95.566667,0.0,-2.566667
2004-01-01 01:30:00,92.200000,0.0,-2.533333
2004-01-01 02:00:00,91.300000,0.0,-2.633333
2004-01-01 02:30:00,92.633333,0.0,-2.800000
...,...,...,...
2018-12-31 22:00:00,99.998000,0.0,3.193900
2018-12-31 22:30:00,99.998000,0.0,3.021733
2018-12-31 23:00:00,99.998000,0.0,2.934467


### Rename variables for merging

In [6]:
# renaming_dict = {
#     'RH_NABEL_T1_49_1': 'RH_T1_47_1',
#     'SW_IN_NABEL_T1_49_1': 'SW_IN_T1_47_1',
#     'TA_NABEL_T1_49_1': 'TA_T1_47_1'
# }
# nabel_diive_ta_rh_swin_2004_2018 = nabel_diive_ta_rh_swin_2004_2018.rename(columns=renaming_dict)
# nabel_diive_ta_rh_swin_2004_2018

</br>

## Data from `mst` meteoscreening (2004-2021)

### Download

In [7]:
%%time

BUCKET = f'ch-lae_processed'
FIELDS = ['TA_T1_47_1', 'RH_T1_47_1', 'SW_IN_T1_47_1']
MEASUREMENTS = ['TA', 'RH', 'SW']
START = '2004-01-01 00:00:01'
STOP = '2022-01-01 00:00:01'
DATA_VERSION = 'meteoscreening_mst'

mst_ta_rh_swin_2004_2021, _, _ = dbc.download(
    bucket=BUCKET,
    measurements=MEASUREMENTS,
    fields=FIELDS,
    start=START,  # Download data starting with this date (the start date itself IS included),
    stop=STOP,  # Download data before this date (the stop date itself IS NOT included),
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version=DATA_VERSION
)


DOWNLOADING
    from bucket ch-lae_processed
    variables ['TA_T1_47_1', 'RH_T1_47_1', 'SW_IN_T1_47_1']
    from measurements ['TA', 'RH', 'SW']
    from data version ['meteoscreening_mst']
    between 2004-01-01 00:00:01 and 2022-01-01 00:00:01
    with timezone offset to UTC of 1
Using querystring:
from(bucket: "ch-lae_processed") |> range(start: 2004-01-01T00:00:01+01:00, stop: 2022-01-01T00:00:01+01:00) |> filter(fn: (r) => r["_measurement"] == "TA" or r["_measurement"] == "RH" or r["_measurement"] == "SW") |> filter(fn: (r) => r["data_version"] == "meteoscreening_mst") |> filter(fn: (r) => r["_field"] == "TA_T1_47_1" or r["_field"] == "RH_T1_47_1" or r["_field"] == "SW_IN_T1_47_1") |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
Used querystring: from(bucket: "ch-lae_processed") |> range(start: 2004-01-01T00:00:01+01:00, stop: 2022-01-01T00:00:01+01:00) |> filter(fn: (r) => r["_measurement"] == "TA" or r["_measurement"] == "RH" or r["_measurement"] == "S

In [9]:
mst_ta_rh_swin_2004_2021

,RH_T1_47_1,SW_IN_T1_47_1,TA_T1_47_1
TIMESTAMP_END,,,
2004-09-20 11:00:00,81.199997,NaN,13.300000
2004-09-20 11:30:00,78.199997,NaN,13.390000
2004-09-20 12:00:00,76.500000,NaN,13.810000
2004-09-20 12:30:00,72.199997,NaN,14.470000
2004-09-20 13:00:00,73.400002,NaN,13.980000
...,...,...,...
2021-12-31 22:00:00,94.843261,-9.935194,7.933211
2021-12-31 22:30:00,93.992424,-10.250348,8.022416
2021-12-31 23:00:00,95.821067,-9.810373,7.719400


### Sanitize timestamp

In [10]:
mst_ta_rh_swin_2004_2021 = TimestampSanitizer(data=mst_ta_rh_swin_2004_2021, output_middle_timestamp=False).get()
mst_ta_rh_swin_2004_2021

,RH_T1_47_1,SW_IN_T1_47_1,TA_T1_47_1
TIMESTAMP_END,,,
2004-09-20 11:00:00,81.199997,NaN,13.300000
2004-09-20 11:30:00,78.199997,NaN,13.390000
2004-09-20 12:00:00,76.500000,NaN,13.810000
2004-09-20 12:30:00,72.199997,NaN,14.470000
2004-09-20 13:00:00,73.400002,NaN,13.980000
...,...,...,...
2021-12-31 22:00:00,94.843261,-9.935194,7.933211
2021-12-31 22:30:00,93.992424,-10.250348,8.022416
2021-12-31 23:00:00,95.821067,-9.810373,7.719400


### Rename variables for merging

In [11]:
# renaming_dict = {
#     'RH_T1_47_1': 'RH_T1_47_1',
#     'SW_IN_T1_47_1': 'SW_IN_T1_47_1',
#     'TA_T1_47_1': 'TA_T1_47_1'
# }
# mst_ta_rh_swin_2019_2021 = mst_ta_rh_swin_2019_2021.rename(columns=renaming_dict)
# mst_ta_rh_swin_2019_2021

</br>

## Data from `diive` meteoscreening (2022-2025)

### Download

In [12]:
%%time

BUCKET = f'ch-lae_processed'
FIELDS = ['TA_T1_47_1', 'RH_T1_47_1', 'SW_IN_T1_47_1']
MEASUREMENTS = ['TA', 'RH', 'SW']
START = '2022-01-01 00:00:01'
STOP = '2026-01-01 00:00:01'
DATA_VERSION = 'meteoscreening_diive'

diive_ta_rh_swin_2022_2025, _, _ = dbc.download(
    bucket=BUCKET,
    measurements=MEASUREMENTS,
    fields=FIELDS,
    start=START,  # Download data starting with this date (the start date itself IS included),
    stop=STOP,  # Download data before this date (the stop date itself IS NOT included),
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version=DATA_VERSION
)


DOWNLOADING
    from bucket ch-lae_processed
    variables ['TA_T1_47_1', 'RH_T1_47_1', 'SW_IN_T1_47_1']
    from measurements ['TA', 'RH', 'SW']
    from data version ['meteoscreening_diive']
    between 2022-01-01 00:00:01 and 2026-01-01 00:00:01
    with timezone offset to UTC of 1
Using querystring:
from(bucket: "ch-lae_processed") |> range(start: 2022-01-01T00:00:01+01:00, stop: 2026-01-01T00:00:01+01:00) |> filter(fn: (r) => r["_measurement"] == "TA" or r["_measurement"] == "RH" or r["_measurement"] == "SW") |> filter(fn: (r) => r["data_version"] == "meteoscreening_diive") |> filter(fn: (r) => r["_field"] == "TA_T1_47_1" or r["_field"] == "RH_T1_47_1" or r["_field"] == "SW_IN_T1_47_1") |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
Used querystring: from(bucket: "ch-lae_processed") |> range(start: 2022-01-01T00:00:01+01:00, stop: 2026-01-01T00:00:01+01:00) |> filter(fn: (r) => r["_measurement"] == "TA" or r["_measurement"] == "RH" or r["_measurement"] =

In [13]:
diive_ta_rh_swin_2022_2025

,RH_T1_47_1,SW_IN_T1_47_1,TA_T1_47_1
TIMESTAMP_END,,,
2022-01-01 00:30:00,89.611768,0.0,8.437600
2022-01-01 01:00:00,89.877390,0.0,8.238294
2022-01-01 01:30:00,90.451803,0.0,8.044655
2022-01-01 02:00:00,89.879497,0.0,8.111550
2022-01-01 02:30:00,89.994291,0.0,7.972055
...,...,...,...
2025-12-31 22:00:00,42.332976,0.0,-3.081644
2025-12-31 22:30:00,39.230344,0.0,-3.084372
2025-12-31 23:00:00,55.771803,0.0,-3.423939


### Sanitize timestamp

In [14]:
diive_ta_rh_swin_2022_2025 = TimestampSanitizer(data=diive_ta_rh_swin_2022_2025, output_middle_timestamp=False).get()
diive_ta_rh_swin_2022_2025

,RH_T1_47_1,SW_IN_T1_47_1,TA_T1_47_1
TIMESTAMP_END,,,
2022-01-01 00:30:00,89.611768,0.0,8.437600
2022-01-01 01:00:00,89.877390,0.0,8.238294
2022-01-01 01:30:00,90.451803,0.0,8.044655
2022-01-01 02:00:00,89.879497,0.0,8.111550
2022-01-01 02:30:00,89.994291,0.0,7.972055
...,...,...,...
2025-12-31 22:00:00,42.332976,0.0,-3.081644
2025-12-31 22:30:00,39.230344,0.0,-3.084372
2025-12-31 23:00:00,55.771803,0.0,-3.423939


### Rename variables for merging

In [15]:
# renaming_dict = {
#     'RH_T1_47_1': 'RH_T1_47_1',
#     'SW_IN_T1_47_1': 'SW_IN_T1_47_1',
#     'TA_T1_47_1': 'TA_T1_47_1'
# }
# diive_ta_rh_swin_2022_2025 = diive_ta_rh_swin_2022_2025.rename(columns=renaming_dict)
# diive_ta_rh_swin_2022_2025

</br>

## Merge data

In [17]:
# Merge data on index
ta_rh_swin_2004_2025 = pd.concat([nabel_diive_ta_rh_swin_2004_2018, mst_ta_rh_swin_2004_2021, diive_ta_rh_swin_2022_2025], axis=0)
ta_rh_swin_2004_2025 = ta_rh_swin_2004_2025.sort_index()
ta_rh_swin_2004_2025

,RH_NABEL_T1_49_1,SW_IN_NABEL_T1_49_1,TA_NABEL_T1_49_1,RH_T1_47_1,SW_IN_T1_47_1,TA_T1_47_1
TIMESTAMP_END,,,,,,
2004-01-01 00:30:00,96.366667,0.0,-2.666667,NaN,NaN,NaN
2004-01-01 01:00:00,95.566667,0.0,-2.566667,NaN,NaN,NaN
2004-01-01 01:30:00,92.200000,0.0,-2.533333,NaN,NaN,NaN
2004-01-01 02:00:00,91.300000,0.0,-2.633333,NaN,NaN,NaN
2004-01-01 02:30:00,92.633333,0.0,-2.800000,NaN,NaN,NaN
...,...,...,...,...,...,...
2025-12-31 22:00:00,NaN,NaN,NaN,42.332976,0.0,-3.081644
2025-12-31 22:30:00,NaN,NaN,NaN,39.230344,0.0,-3.084372
2025-12-31 23:00:00,NaN,NaN,NaN,55.771803,0.0,-3.423939


</br>

### Sanitize timestamp

In [18]:
ta_rh_swin_2004_2025 = TimestampSanitizer(data=ta_rh_swin_2004_2025, output_middle_timestamp=False).get()
ta_rh_swin_2004_2025

,RH_NABEL_T1_49_1,SW_IN_NABEL_T1_49_1,TA_NABEL_T1_49_1,RH_T1_47_1,SW_IN_T1_47_1,TA_T1_47_1
TIMESTAMP_END,,,,,,
2004-01-01 00:30:00,96.366667,0.0,-2.666667,NaN,NaN,NaN
2004-01-01 01:00:00,95.566667,0.0,-2.566667,NaN,NaN,NaN
2004-01-01 01:30:00,92.200000,0.0,-2.533333,NaN,NaN,NaN
2004-01-01 02:00:00,91.300000,0.0,-2.633333,NaN,NaN,NaN
2004-01-01 02:30:00,92.633333,0.0,-2.800000,NaN,NaN,NaN
...,...,...,...,...,...,...
2025-12-31 22:00:00,NaN,NaN,NaN,42.332976,0.0,-3.081644
2025-12-31 22:30:00,NaN,NaN,NaN,39.230344,0.0,-3.084372
2025-12-31 23:00:00,NaN,NaN,NaN,55.771803,0.0,-3.423939


</br>

### Correction: Remove zero offset < 0 from `SW_IN`

In [ ]:
_swin = ta_rh_swin_2004_2025['SW_IN_T1_47_1'].copy()
_swin_corrected = remove_radiation_zero_offset(series=_swin, lat=SITE_LAT, lon=SITE_LON, utc_offset=1, showplot=True)
ta_rh_swin_2004_2025['SW_IN_T1_47_1'] = np.nan
ta_rh_swin_2004_2025['SW_IN_T1_47_1'] = _swin_corrected

</br>

### Correction: Remove offset >100% from `RH`

In [ ]:
_rh = ta_rh_swin_2004_2025['RH_T1_47_1'].copy()
_rh_corrected = remove_relativehumidity_offset(series=_rh, showplot=True)
ta_rh_swin_2004_2025['RH_T1_47_1'] = np.nan
ta_rh_swin_2004_2025['RH_T1_47_1'] = _rh_corrected

</br>

## Dataframe

In [ ]:
ta_rh_swin_2004_2025

</br>

## Plot heatmaps

In [ ]:
fig, axs = plt.subplots(ncols=3, figsize=(14, 10), dpi=100, layout="constrained")
fig.suptitle(f'Half-hourly', fontsize=16)
dv.heatmapdatetime(series=ta_rh_swin_2004_2025['SW_IN_T1_47_1'], title="SW_IN_T1_47_1", ax=axs[0], cb_digits_after_comma=0, zlabel="value").plot()
dv.heatmapdatetime(series=ta_rh_swin_2004_2025['TA_T1_47_1'], title="TA_T1_47_1", ax=axs[1], cb_digits_after_comma=0, zlabel="value").plot()
dv.heatmapdatetime(series=ta_rh_swin_2004_2025['RH_T1_47_1'], title="RH_T1_47_1", ax=axs[2], cb_digits_after_comma=0, zlabel="value").plot()

</br>

</br>

# Save to file

In [ ]:
OUTNAME = "02_METEO6_NOT-GAPFILLED_2004-2025"
OUTPATH = r""
filepath = save_parquet(filename=OUTNAME, data=merged_df, outpath=OUTPATH)
merged_df.to_csv(Path(OUTPATH) / f"{OUTNAME}.csv")

</br>

# End of notebook.

In [ ]:
dt_string = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Finished. {dt_string}")

</br>